# Large Brain Model (LaBraM)

This notebook trains and evaluates the Large Brain Model (LaBraM) using pretrained foundation layers and LoRA adapters to decode binocular rivalry states (Congruent vs. Incongruent Pure).

### Large Brain Model (LaBraM) Downstream Adaptation Pipeline

LaBraM is pre-trained on massive EEG datasets. Here we adapt it to binary decoding with LoRA updates on self-attention projections, using a neural tokenizer and a summary token classification head.

```mermaid
graph TD
    A[Input Epochs: batch, channels, times] --> B[Reshape to batch*channels, 1, times]
    B --> C[Neural Tokenizer: Conv1D -> GELU -> Conv1D -> GELU]
    C --> D[Add Spatial & Temporal Positional Embeddings]
    D --> E[Flatten channels & patches to sequence]
    E --> F[Prepend Summary Token: summary_token]
    F --> G[LaBraM Transformer Blocks with LoRA adapters]
    G --> H[Extract summary_token state & LayerNorm]
    H --> I[Fully Connected Head with Dropout]
    I --> J[Classification Output: Congruent vs Incongruent Pure]
```

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'

import sys
import warnings
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
%pip install -q mne braindecode[hub] timm safetensors huggingface_hub shap
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, auc
from braindecode.models import InterpolatedLaBraM

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (
    Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if "google.colab" in sys.modules
    else NOTEBOOK_DIR.parent.parent / "src"
)

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import (
    BinaryTaskConfig,
    RawEpochSpec,
    detect_project_paths,
    load_raw_epoch_dataset,
    get_task_config
)
from br_analysis.pretrained import (
    BackboneClassifier,
    prepare_lora_backbone,
    load_backbone_weights
)
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
)
from br_analysis.torch_models import count_trainable_parameters

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")

## Task And Environment Setup

In [ ]:
TASK_TAG = "c_vs_ip_labram_pretrained_lora"
paths = detect_project_paths("c_vs_i_labram_pretrained_lora", in_colab=IN_COLAB)
SELECTED_SUBJECTS_CSV = paths.analysis_root / "exploration" / "tables" / "subjectes_seleccionats.csv"
MAX_SUBJECTS = 24

task = get_task_config("c_vs_ip", paths.analysis_root, name=TASK_TAG, analysis_window=(0.0, 1.5))

print(f"Project: {paths.project_dir}")
print(f"Data: {paths.data_dir}")
print(f"Results: {paths.analysis_dir}")

## Load Epochs And Pad Layout

In [ ]:
spec = RawEpochSpec(picks=None, bandpass=(1.0, 40.0), resample_sfreq=128.0)
dataset = load_raw_epoch_dataset(paths, task, spec)
X_raw = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y, dtype=np.int64)
subjects = np.asarray(dataset.subjects, dtype=object)
SUBJECT_ID = "P057"
ch_names = list(dataset.channel_names)
times = dataset.times
chs_info = dataset.chs_info
sfreq = 128.0

# Use native epoch length
TARGET_LEN = X_raw.shape[-1]
X = X_raw

print(f"X shape: {X.shape} (trials, channels, times)")
print(f"Classes: {np.bincount(y)}")

## LaBraM Model Definition & Configuration

In [ ]:
def make_model():
    ch_names = [ch["ch_name"] for ch in chs_info]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    montage = mne.channels.make_standard_montage("standard_1005")
    info.set_montage(montage)

    model = InterpolatedLaBraM.from_pretrained(
        "braindecode/labram-pretrained",
        chs_info=info["chs"],
        n_times=TARGET_LEN,
        n_outputs=2,
        patch_size=200,
        strict=False,
    )

    prepare_lora_backbone(
        model,
        include_substrings=("attn",),
        exclude_substrings=("head", "classifier", "interpolation_layer"),
        r=8,
        alpha=16.0,
        dropout=0.05,
        train_classifier=True,
    )

    if hasattr(model, "get_classifier"):
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    return model


train_cfg = TorchTrainConfig(
    epochs=50,
    batch_size=8,
    lr=2e-4,
    weight_decay=1e-4,
    patience=10,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
    progress=True,
)

## 3. Cohort-Wide Training (LoRA Fine-tuning)

We train the LoRA fine-tuned LaBraM model across all 24 subjects individually and cache the results.

In [ ]:
cohort_metrics_path = paths.tables_dir / f"cohort_cv_metrics_{TASK_TAG}.csv"
cohort_fold_metrics_path = paths.tables_dir / f"cohort_fold_metrics_{TASK_TAG}.csv"
subject_ids = pd.unique(subjects).tolist()

if cohort_metrics_path.exists() and cohort_fold_metrics_path.exists():
    print("Loading LaBraM cohort metrics from cache...")
    df_cohort = pd.read_csv(cohort_metrics_path)
    df_cohort_folds = pd.read_csv(cohort_fold_metrics_path)
else:
    print("Running LaBraM cohort-wide cross-validation with LoRA adapters...")
    cohort_results = []
    cohort_fold_results = []

    silent_cfg = TorchTrainConfig(
        epochs=train_cfg.epochs,
        batch_size=train_cfg.batch_size,
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
        patience=train_cfg.patience,
        n_splits=train_cfg.n_splits,
        val_size=train_cfg.val_size,
        random_seed=train_cfg.random_seed,
        device=train_cfg.device,
        progress=False,
    )

    for sub in subject_ids:
        print(f"Training subject {sub}...")
        labram_metrics, _ = cross_validate_torch_model(
            X,
            y,
            make_model,
            silent_cfg,
            subjects=subjects,
            subject_id=sub,
            save_weights_dir=paths.cache_dir,
            weights_prefix=f"{TASK_TAG}_sub{sub}",
            load_weights=True,
        )

        cohort_results.append(
            {
                "subject": sub,
                "labram_acc": labram_metrics["test_acc"].mean(),
                "labram_auc": labram_metrics["test_auc"].mean(),
                "labram_bal_acc": labram_metrics["test_balanced_acc"].mean(),
            }
        )

        for idx, row in labram_metrics.iterrows():
            cohort_fold_results.append(
                {
                    "subject": sub,
                    "model": "LaBraM",
                    "fold": int(row["fold"]),
                    "test_acc": row["test_acc"],
                    "test_auc": row["test_auc"],
                    "test_balanced_acc": row["test_balanced_acc"],
                }
            )

    df_cohort = pd.DataFrame(cohort_results)
    df_cohort.to_csv(cohort_metrics_path, index=False)
    df_cohort_folds = pd.DataFrame(cohort_fold_results)
    df_cohort_folds.to_csv(cohort_fold_metrics_path, index=False)
    print(f"Saved cohort metrics to: {cohort_metrics_path}")

print(df_cohort.describe())

## 4. Single-Subject Detailed Performance Comparison (P057)

In [ ]:
from br_analysis.torch_training import build_dataloader, standardize_train_test, _evaluate, temporal_stratified_splits

device = torch.device(DEVICE)
splits = list(temporal_stratified_splits(y[subjects == SUBJECT_ID], train_cfg.n_splits))
X_sub = X[subjects == SUBJECT_ID]
y_sub = y[subjects == SUBJECT_ID]

probs = []
y_true_all = []

for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
    y_test = y_sub[test_idx]
    y_true_all.extend(y_test)

    model = make_model().to(device)
    model.load_state_dict(
        torch.load(paths.cache_dir / f"{TASK_TAG}_sub{SUBJECT_ID}_fold{fold}.pt", map_location=device)
    )
    loader = build_dataloader(X_test, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
    _, _, _, p = _evaluate(model, loader, device)
    probs.extend(p)

y_true_all = np.array(y_true_all)
probs = np.array(probs)
preds = (probs >= 0.5).astype(int)

print("=== LaBraM Classification Report (Subject P057) ===")
print(classification_report(y_true_all, preds, target_names=[task.class_0, task.class_1]))

# Plot ROC/PR curves
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(y_true_all, probs)
ax_roc.plot(fpr, tpr, label=f"LaBraM (AUC = {auc(fpr, tpr):.3f})", color="purple", linewidth=2)
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax_roc.set_title("ROC Curve (Subject P057)")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend()

prec, rec, _ = precision_recall_curve(y_true_all, probs)
ax_pr.plot(rec, prec, label=f"LaBraM (μP = {auc(rec, prec):.3f})", color="purple", linewidth=2)
ax_pr.plot([0, 1], [0.5, 0.5], "k--", alpha=0.5)
ax_pr.set_title("Precision-Recall Curve (Subject P057)")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()

plt.tight_layout()
plt.savefig(paths.figures_dir / f"roc_pr_{TASK_TAG}_P057.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Single-Subject Fold Analysis & Saliency Mapping (P057)

We statistically evaluate the 5 CV folds against the 50% chance level and extract absolute gradients to map spatial and temporal importances.

In [ ]:
from scipy.stats import ttest_1samp, wilcoxon

metrics_sub = df_cohort_folds[df_cohort_folds["subject"] == SUBJECT_ID].copy()
acc_folds = metrics_sub["test_acc"].values

t_stat, p_val_t = ttest_1samp(acc_folds, 0.5)
w_stat, p_val_w = wilcoxon(acc_folds - 0.5)

print("=== P057 Fold Accuracies ===")
print(f"Accuracies: {acc_folds}")
print(f"One-sample t-test (vs 0.5): t-statistic = {t_stat:.3f}, p-value = {p_val_t:.4f}")
print(f"One-sample Wilcoxon (vs 0.5): W-statistic = {w_stat:.3f}, p-value = {p_val_w:.4f}")

plt.figure(figsize=(4, 5))
sns.boxplot(y=acc_folds, color="mediumpurple", width=0.4)
sns.stripplot(y=acc_folds, color="black", size=8, jitter=0.05)
plt.axhline(0.5, color="black", linestyle=":", alpha=0.5, label="Chance")
plt.title(f"LaBraM Folds (P057)\nt-test p = {p_val_t:.4f}")
plt.ylabel("Classification Accuracy")
plt.ylim(0.2, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig(paths.figures_dir / f"fold_accuracy_boxplot_P057_{TASK_TAG}.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Compute P057 Channel/Temporal Saliency (gradients w.r.t input channels/time)
device = torch.device(DEVICE)
splits = list(temporal_stratified_splits(y[subjects == SUBJECT_ID], train_cfg.n_splits))
X_sub = X[subjects == SUBJECT_ID]
y_sub = y[subjects == SUBJECT_ID]

saliencies = []
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
    y_test = y_sub[test_idx]

    # Filter conflict trials (class 1)
    idx_c1 = y_test == 1
    if not np.any(idx_c1):
        continue

    model = make_model().to(device)
    model.load_state_dict(
        torch.load(paths.cache_dir / f"{TASK_TAG}_sub{SUBJECT_ID}_fold{fold}.pt", map_location=device)
    )
    model.eval()

    X_tensor = torch.from_numpy(X_test[idx_c1]).to(device)
    X_tensor.requires_grad_()
    logits = model(X_tensor)
    probs_out = torch.softmax(logits, dim=1)
    score = probs_out[:, 1].sum()
    model.zero_grad()
    score.backward()

    saliency = X_tensor.grad.abs().cpu().numpy()  # Shape: (trials, 60, times)
    saliencies.append(saliency.mean(axis=0))  # Mean over trials

mean_saliency_p057 = np.mean(saliencies, axis=0)  # Shape: (60, times)

spatial_imp = mean_saliency_p057.mean(axis=1)  # Shape: (60,)
temporal_imp = mean_saliency_p057.mean(axis=0)  # Shape: (times,)

# Plot Spatial Node Saliency
fig, ax = plt.subplots(figsize=(5, 5))
montage = mne.channels.make_standard_montage("standard_1005")
info = mne.create_info(ch_names=ch_names, sfreq=128.0, ch_types="eeg")
info.set_montage(montage, on_missing="ignore")
im, _ = mne.viz.plot_topomap(spatial_imp, info, axes=ax, cmap="viridis", show=False)
ax.set_title("LaBraM Spatial Node Saliency (P057)")
fig.colorbar(im, ax=ax, shrink=0.6, label="Mean gradient magnitude")
plt.tight_layout()
plt.savefig(paths.figures_dir / f"saliency_spatial_P057_{TASK_TAG}.png", dpi=180, bbox_inches="tight")
plt.show()

# Plot Temporal Saliency Profile (zoom in on [0, 1.5] seconds stimulus window)
plt.figure(figsize=(7, 4))
t_sec = np.arange(TARGET_LEN) / sfreq
plt.plot(t_sec, temporal_imp, color="purple", linewidth=2)
plt.axvline(0.0, color="black", alpha=0.5, linestyle=":")
plt.title("LaBraM Temporal Saliency Profile (P057)")
plt.xlabel("Time post-stimulus (s)")
plt.ylabel("Mean Saliency Gradient Magnitude")
plt.xlim(0.0, 1.5)
plt.legend()
plt.grid(True, linestyle=":")
plt.tight_layout()
plt.savefig(paths.figures_dir / f"saliency_temporal_P057_{TASK_TAG}.png", dpi=180, bbox_inches="tight")
plt.show()

## 5.2 SHAP (SHapley Additive exPlanations) Explainability

We use SHAP (SHapley Additive exPlanations) via `shap.GradientExplainer` to compute feature attributions for electrode channels and time samples in the pre-trained LaBraM model.

In [ ]:
# Compute P057 SHAP values
import shap

device = torch.device(DEVICE)
splits = list(temporal_stratified_splits(y[subjects == SUBJECT_ID], train_cfg.n_splits))
X_sub = X[subjects == SUBJECT_ID]
y_sub = y[subjects == SUBJECT_ID]

# Use fold 1 model for SHAP
train_idx, test_idx = splits[0]
_, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
y_test = y_sub[test_idx]

# Background distribution from test set (first 50 trials for speed)
bg_tensor = torch.from_numpy(X_test[:50]).to(device)
test_tensor = torch.from_numpy(X_test[y_test == 1]).to(device)  # Explain conflict trials

model = make_model().to(device)
model.load_state_dict(torch.load(paths.cache_dir / f"{TASK_TAG}_sub{SUBJECT_ID}_fold1.pt", map_location=device))
model.eval()

# Explain conflict class (class 1)
explainer = shap.GradientExplainer(model, bg_tensor)
shap_values = explainer.shap_values(test_tensor)

if isinstance(shap_values, list):
    shap_class1 = shap_values[1]
else:
    shap_class1 = shap_values

# shap_class1 shape: (trials, 60, 3000)
shap_spatial = np.abs(shap_class1).mean(axis=(0, 2))  # shape (60,)
shap_temporal = np.abs(shap_class1).mean(axis=(0, 1))  # shape (3000,)

# Plot spatial SHAP topomap
fig, (ax_spat, ax_temp) = plt.subplots(1, 2, figsize=(10, 4.5))
montage = mne.channels.make_standard_montage("standard_1005")
info = mne.create_info(ch_names=ch_names, sfreq=128.0, ch_types="eeg")
info.set_montage(montage, on_missing="ignore")
im, _ = mne.viz.plot_topomap(shap_spatial, info, axes=ax_spat, cmap="viridis", show=False)
ax_spat.set_title("LaBraM SHAP Spatial (P057)")
fig.colorbar(im, ax=ax_spat, shrink=0.6, label="Mean SHAP magnitude")

# Plot temporal SHAP profile
t_sec = np.arange(TARGET_LEN) / sfreq
ax_temp.plot(t_sec, shap_temporal, color="purple", linewidth=2)
ax_temp.set_title("LaBraM SHAP Temporal (P057)")
ax_temp.set_xlabel("Time post-stimulus (s)")
ax_temp.set_ylabel("Mean SHAP magnitude")
ax_temp.set_xlim(0.0, 1.5)
ax_temp.legend()
ax_temp.grid(True, linestyle=":")

plt.tight_layout()
plt.savefig(paths.figures_dir / f"shap_explainability_P057_{TASK_TAG}.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Cohort-Level Group Performance, Error Asymmetry & Saliency

In [ ]:
# Plot Group Performance Boxplot
t_stat, p_val_t = ttest_1samp(df_cohort["labram_acc"], 0.5)
w_stat, p_val_w = wilcoxon(df_cohort["labram_acc"] - 0.5)

print("=== Cohort Performance Stats (vs 0.5) ===")
print(f"Mean Acc: {df_cohort['labram_acc'].mean():.4f} | Std Acc: {df_cohort['labram_acc'].std():.4f}")
print(f"t-test: t-stat={t_stat:.3f}, p-val={p_val_t:.4f}")
print(f"Wilcoxon: W-stat={w_stat:.3f}, p-val={p_val_w:.4f}")

plt.figure(figsize=(5, 5.5))
sns.boxplot(y=df_cohort["labram_acc"], color="mediumpurple", width=0.4)
sns.stripplot(y=df_cohort["labram_acc"], color="black", size=6, jitter=0.08)
plt.axhline(0.5, color="black", linestyle=":", alpha=0.5, label="Chance")
plt.title(f"LaBraM Cohort Performance (N={len(df_cohort)})\nt-test p = {p_val_t:.4f} | Wilcoxon p = {p_val_w:.4f}")
plt.ylabel("Classification Accuracy")
plt.ylim(0.3, 0.95)
plt.legend()
plt.tight_layout()
plt.savefig(paths.figures_dir / f"cohort_performance_boxplot_{TASK_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compute cohort-wide aggregated ROC/PR curves
all_y_true = []
all_probs = []
subject_y_trues = {}
subject_y_probs = {}

print("Extracting cohort predictions across all subjects and folds...")
for sub in subject_ids:
    splits = list(temporal_stratified_splits(y[subjects == sub], train_cfg.n_splits))
    X_sub = X[subjects == sub]
    y_sub = y[subjects == sub]

    sub_y_true = []
    sub_y_prob = []

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
        y_test = y_sub[test_idx]

        weights_path = paths.cache_dir / f"{TASK_TAG}_sub{sub}_fold{fold}.pt"
        if weights_path.exists():
            model = make_model().to(device)
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model.eval()

            loader = build_dataloader(X_test, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
            _, _, y_true_arr, p = _evaluate(model, loader, device)
            all_y_true.extend(y_true_arr)
            all_probs.extend(p)
            sub_y_true.extend(y_true_arr)
            sub_y_prob.extend(p)

    if len(sub_y_true) > 0:
        subject_y_trues[sub] = np.array(sub_y_true)
        subject_y_probs[sub] = np.array(sub_y_prob)

all_y_true = np.array(all_y_true)
all_probs = np.array(all_probs)
all_preds = (all_probs >= 0.5).astype(int)

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(all_y_true, all_probs)
ax_roc.plot(fpr, tpr, label=f"LaBraM (AUC = {auc(fpr, tpr):.3f})", color="purple", linewidth=2)
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax_roc.set_title(f"Cohort ROC Curve (N={len(subject_ids)})")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend()

prec, rec, _ = precision_recall_curve(all_y_true, all_probs)
ax_pr.plot(rec, prec, label=f"LaBraM (μP = {auc(rec, prec):.3f})", color="purple", linewidth=2)
ax_pr.plot([0, 1], [0.5, 0.5], "k--", alpha=0.5)
ax_pr.set_title(f"Cohort Precision-Recall Curve (N={len(subject_ids)})")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()

plt.tight_layout()
plt.savefig(paths.figures_dir / f"cohort_roc_pr_{TASK_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 1. Standard aggregated reports (all folds combined)
reports = []
for sub in subject_y_trues.keys():
    sub_y_true = subject_y_trues[sub]
    sub_y_pred = (subject_y_probs[sub] >= 0.5).astype(int)
    rep = classification_report(sub_y_true, sub_y_pred, target_names=[task.class_0, task.class_1], output_dict=True)
    reports.append(rep)

avg_report = {}
metrics_keys = ["precision", "recall", "f1-score", "support"]
class_keys = [task.class_0, task.class_1, "macro avg", "weighted avg"]
for ck in class_keys:
    avg_report[ck] = {}
    for mk in metrics_keys:
        avg_report[ck][mk] = np.mean([r[ck][mk] for r in reports])
avg_report["accuracy"] = np.mean([r["accuracy"] for r in reports])

print("==========================================================")
print("=== AVERAGE CLASSIFICATION REPORT (ALL FOLDS COMBINED) ===")
print("==========================================================")
print("--- LaBraM (All Folds) ---")
print(pd.DataFrame(avg_report).T)


# 2. Extract and compute classification reports specifically for FOLD 5 (representing maximum training data)
print("\n\n" + "=" * 58)
print("=== FOLD 5 CLASSIFICATION REPORT (MAX TRAINING TRIALS) ===")
print("=" * 58)

reports_f5 = []

for sub in subject_ids:
    splits = list(temporal_stratified_splits(y[subjects == sub], train_cfg.n_splits))
    X_sub = X[subjects == sub]
    y_sub = y[subjects == sub]

    # We focus on Fold 5 (index 4 of the splits)
    train_idx, test_idx = splits[4]

    _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
    y_test = y_sub[test_idx]

    # Fold 5 weights
    weights_path = paths.cache_dir / f"{TASK_TAG}_sub{sub}_fold5.pt"
    if weights_path.exists():
        model = make_model().to(device)
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.eval()
        loader = build_dataloader(X_test, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
        _, _, y_true_arr, p = _evaluate(model, loader, device)
        rep = classification_report(
            y_true_arr, (p >= 0.5).astype(int), target_names=[task.class_0, task.class_1], output_dict=True
        )
        reports_f5.append(rep)

avg_report_f5 = {}
for ck in class_keys:
    avg_report_f5[ck] = {}
    for mk in metrics_keys:
        avg_report_f5[ck][mk] = np.mean([r[ck][mk] for r in reports_f5])
avg_report_f5["accuracy"] = np.mean([r["accuracy"] for r in reports_f5])

print("--- LaBraM (Fold 5) ---")
print(pd.DataFrame(avg_report_f5).T)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_y_true, all_preds, normalize="true")
tn, fp, fn, tp = confusion_matrix(all_y_true, all_preds).ravel()
bias_ratio = fp / fn if fn > 0 else np.nan

print("=== LaBraM Cohort Bias Analysis ===")
print(f"TN: {tn} | FP: {fp} | FN: {fn} | TP: {tp}")
print(f"FP / FN Bias Ratio: {bias_ratio:.3f}")

plt.figure(figsize=(5.5, 4.5))
classes = [task.class_0, task.class_1]
sns.heatmap(cm, annot=True, fmt=".3f", cmap="Purples", xticklabels=classes, yticklabels=classes, cbar=False)
plt.title("LaBraM Cohort-Wide Normalized Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.savefig(paths.figures_dir / f"cohort_confusion_matrix_{TASK_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import ttest_1samp
from mne.stats import fdr_correction

group_diff_spatial = []
print("Computing cohort-wide differential saliency maps (Class 1 - Class 0)... ")
for sub in subject_ids:
    splits = list(temporal_stratified_splits(y[subjects == sub], train_cfg.n_splits))
    X_sub = X[subjects == sub]
    y_sub = y[subjects == sub]

    sub_c1 = []
    sub_c0 = []

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
        y_test = y_sub[test_idx]

        weights_path = paths.cache_dir / f"{TASK_TAG}_sub{sub}_fold{fold}.pt"
        if weights_path.exists():
            model = make_model().to(device)
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model.eval()

            # Class 1
            idx_c1 = y_test == 1
            if np.any(idx_c1):
                X_tensor = torch.from_numpy(X_test[idx_c1]).to(device)
                X_tensor.requires_grad_()
                logits = model(X_tensor)
                probs_out = torch.softmax(logits, dim=1)
                score = probs_out[:, 1].sum()
                model.zero_grad()
                score.backward()
                saliency_c1 = X_tensor.grad.abs().cpu().numpy().mean(axis=0)  # shape (60, times)
                sub_c1.append(saliency_c1)

            # Class 0
            idx_c0 = y_test == 0
            if np.any(idx_c0):
                X_tensor = torch.from_numpy(X_test[idx_c0]).to(device)
                X_tensor.requires_grad_()
                logits = model(X_tensor)
                probs_out = torch.softmax(logits, dim=1)
                score = probs_out[:, 0].sum()
                model.zero_grad()
                score.backward()
                saliency_c0 = X_tensor.grad.abs().cpu().numpy().mean(axis=0)
                sub_c0.append(saliency_c0)

    if sub_c1 and sub_c0:
        mean_c1 = np.mean(sub_c1, axis=0)
        mean_c0 = np.mean(sub_c0, axis=0)

        mean_c1_norm = mean_c1 / np.max(mean_c1) if np.max(mean_c1) > 0 else mean_c1
        mean_c0_norm = mean_c0 / np.max(mean_c0) if np.max(mean_c0) > 0 else mean_c0

        diff = mean_c1_norm - mean_c0_norm
        spatial_diff = np.mean(diff, axis=1)  # average over times, shape (60,)
        group_diff_spatial.append(spatial_diff)

if group_diff_spatial:
    group_diff_spatial = np.array(group_diff_spatial)
    n_channels = group_diff_spatial.shape[1]
    t_stats = np.zeros(n_channels)
    p_vals = np.zeros(n_channels)

    for ch in range(n_channels):
        t_stat, p_val = ttest_1samp(group_diff_spatial[:, ch], 0.0)
        t_stats[ch] = t_stat
        p_vals[ch] = p_val

    reject, p_vals_fdr = fdr_correction(p_vals, alpha=0.05, method="indep")
    print(f"FDR Significant Channels (p_FDR < 0.05): {[ch_names[i] for i, val in enumerate(reject) if val]}")

    # Plot t-stats differential topomap
    fig, ax = plt.subplots(figsize=(6, 5))
    montage = mne.channels.make_standard_montage("standard_1005")
    info = mne.create_info(ch_names=ch_names, sfreq=128.0, ch_types="eeg")
    info.set_montage(montage, on_missing="ignore")

    im, _ = mne.viz.plot_topomap(
        t_stats,
        info,
        axes=ax,
        mask=reject,
        mask_params=dict(marker="o", markerfacecolor="black", markersize=6, markeredgecolor="black"),
        cmap="RdBu_r",
        vlim=(-4.0, 4.0),
        show=False,
    )

    ax.set_title(
        "Group-Level Statistical Differential Saliency Map\n"
        "(Saliency Class 1 - Class 0 | One-sample t-test | masked at p_FDR < 0.05)",
        fontsize=10,
        fontweight="bold",
    )
    fig.colorbar(im, ax=ax, shrink=0.6, label="t-statistic")
    plt.tight_layout()
    plt.savefig(paths.figures_dir / f"group_saliency_differential_topomap_{TASK_TAG}.png", dpi=220, bbox_inches="tight")
    plt.show()

In [ ]:
# Compute cohort-wide temporal saliency profiles
group_temp_saliencies = []
for sub in subject_ids:
    splits = list(temporal_stratified_splits(y[subjects == sub], train_cfg.n_splits))
    X_sub = X[subjects == sub]
    y_sub = y[subjects == sub]
    sub_temp = []

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        _, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
        y_test = y_sub[test_idx]

        weights_path = paths.cache_dir / f"{TASK_TAG}_sub{sub}_fold{fold}.pt"
        if weights_path.exists():
            model = make_model().to(device)
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model.eval()

            idx_c1 = y_test == 1
            if np.any(idx_c1):
                X_tensor = torch.from_numpy(X_test[idx_c1]).to(device)
                X_tensor.requires_grad_()
                logits = model(X_tensor)
                probs_out = torch.softmax(logits, dim=1)
                score = probs_out[:, 1].sum()
                model.zero_grad()
                score.backward()
                saliency = X_tensor.grad.abs().cpu().numpy().mean(axis=0)  # shape (60, times)
                sub_temp.append(saliency)

    if sub_temp:
        mean_sub = np.mean(sub_temp, axis=0)  # Shape: (60, times)
        mean_sub_norm = mean_sub / np.max(mean_sub) if np.max(mean_sub) > 0 else mean_sub
        group_temp_saliencies.append(mean_sub_norm.mean(axis=0))  # Average over channels, shape (times,)

if group_temp_saliencies:
    group_temp_saliencies = np.array(group_temp_saliencies)
    mean_temp_sal = np.mean(group_temp_saliencies, axis=0)
    t_sec = np.arange(TARGET_LEN) / sfreq

    plt.figure(figsize=(7, 4))
    plt.plot(t_sec, mean_temp_sal, color="purple", linewidth=2.2)
    plt.axvline(0.0, color="black", alpha=0.5, linestyle=":")
    plt.title(f"Group-Average LaBraM Temporal Saliency (N={len(group_temp_saliencies)})")
    plt.xlabel("Time post-stimulus (s)")
    plt.ylabel("Mean Saliency Gradient Magnitude")
    plt.xlim(0.0, 1.5)
    plt.legend()
    plt.grid(True, linestyle=":")
    plt.tight_layout()
    plt.savefig(paths.figures_dir / f"group_saliency_temporal_{TASK_TAG}.png", dpi=200, bbox_inches="tight")
    plt.show()